# Naive Bayes

***

## Importando bibliotecas

In [ ]:
%pip install plotly nbformat


In [3]:
# Bibliotecas de manipulação e visualização de dados
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde

# Classes do modelo de aprendizado
from sklearn.naive_bayes import GaussianNB, BernoulliNB, MultinomialNB

# Funções de avaliação dos modelos
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')


In [ ]:
#Carregando o dataset
dataset = pd.read_csv('Datasets\Iris.csv')
dataset

,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,virginica
146,6.3,2.5,5.0,1.9,virginica
147,6.5,3.0,5.2,2.0,virginica
148,6.2,3.4,5.4,2.3,virginica


In [5]:
#Mapeando os valores da classe para inteiro (para fins de visualização da região de decisão)
dataset['Species'] = pd.factorize(dataset['Species'])[0]

### Separando o conjunto de dados

In [6]:
#Vamos usar somente duas features PetalLengthCm e PetalWidthCm 
X = dataset.iloc[:,[2,3]]
y = dataset.iloc[:,[4]]

#Separando o conjunto de dados em treinamento e teste
X_train, X_test, y_train, y_test = train_test_split(X, y)

## Distribuição dos dados

Saber como as nossas variáveis se comportam é essencial para o classificador bayesiano, já que o classificador utiliza a distribuição de probabilidade $P(X|y)$ para fazer a classificação, precisamos entender o comportamento da distribuição para saber qual família de classificadores bayesianos iremos aplicar. 

Cada classificador utiliza uma suposição diferente da função $P(X|y)$:

[CategoricalNB](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.CategoricalNB.html#sklearn.naive_bayes.CategoricalNB) - Implementa o algoritmo de Naive Bayes categórico para dados distribuídos categoricamente.

[GaussianNB](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html#sklearn.naive_bayes.GaussianNB) - Implementa o algoritmo Gaussian Naive Bayes para classificação. A probabilidade das features é assumida como gaussiana.

[MultinomialNB](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html#sklearn.naive_bayes.MultinomialNB) - Implementa o algoritmo Naive Bayes para dados multinomialmente distribuídos e é uma das duas variantes clássicas Naive Bayes usadas na classificação de texto.
A distribuição multinomial modela o número de vezes que ocorrem categorias específicas em um conjunto.

[BernoulliNB](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.BernoulliNB.html#sklearn.naive_bayes.BernoulliNB) - Implementa o algoritmo Naive Bayes para dados que são distribuídos de acordo com distribuições multivariadas de Bernoulli; ou seja, pode haver várias features, mas cada um é assumido como uma variável de valor binário.

Mais detalhes [link](https://scikit-learn.org/stable/modules/naive_bayes.html#).

Como selecionar:

- Features categóricas: CategoricalNB
- Features contínuas: GuassianNB
- Distribuições com features discretas, ou uma contagem (0, 1, 2, 3, ...): MultinomialNB ou ComplementNB
- Para features como distribuições de Bernoulli: BernoulliNB

### Representação gráfica da distribuição

O gráfico de estimativa de densidade de kernel (***kernel density estimate*** - KDE) é um método para visualizar a distribuição de observações em um conjunto de dados, análogo a um histograma. O KDE representa os dados usando uma curva de densidade de probabilidade contínua em uma ou mais dimensões.


In [7]:
plot_data = X.copy()
plot_data["Class"] = y.astype(str)

fig = make_subplots(rows=1, cols=2, subplot_titles=["PetalLengthCm", "PetalWidthCm"])
colors = px.colors.qualitative.Plotly

for col_idx, feature in enumerate(["PetalLengthCm", "PetalWidthCm"], start=1):
    for j, cls in enumerate(sorted(plot_data["Class"].unique())):
        values = plot_data.loc[plot_data["Class"] == cls, feature].dropna()
        xs = np.linspace(values.min() - 0.2, values.max() + 0.2, 200)
        kde = gaussian_kde(values)
        fig.add_trace(
            go.Scatter(
                x=xs,
                y=kde(xs),
                mode="lines",
                name=f"Class {cls}",
                legendgroup=cls,
                showlegend=(col_idx == 1),
                line=dict(color=colors[j % len(colors)]),
            ),
            row=1,
            col=col_idx,
        )

fig.update_layout(
    title="Estimativa de densidade (KDE)",
    width=800,
    height=500,
)
fig.show()


No nosso caso, as distribuições se assemelham a uma distribuição normal. Portanto vamos utilizar o $GuassianNaiveBayes$.

In [8]:
def show_decision_region(X, y, model):
    feature_0 = X.columns[0]
    feature_1 = X.columns[1]

    x_min, x_max = X[feature_0].min() - 0.5, X[feature_0].max() + 0.5
    y_min, y_max = X[feature_1].min() - 0.5, X[feature_1].max() + 0.5

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200),
    )
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    fig = go.Figure()
    fig.add_trace(go.Contour(
        x=np.linspace(x_min, x_max, 200),
        y=np.linspace(y_min, y_max, 200),
        z=Z,
        showscale=False,
        opacity=0.45,
        colorscale="Viridis",
        contours=dict(showlines=False),
    ))
    fig.add_trace(go.Scatter(
        x=X[feature_0],
        y=X[feature_1],
        mode="markers",
        marker=dict(
            color=y.values.flatten(),
            colorscale="Viridis",
            size=8,
            line=dict(width=1, color="white"),
        ),
        name="Pontos",
    ))
    fig.update_layout(
        title=model.__class__.__name__,
        xaxis_title=feature_0,
        yaxis_title=feature_1,
        width=600,
        height=500,
        showlegend=False,
    )
    fig.show()


### Treinamento do modelo

In [9]:
# Comportamento de diferentes modelos 

names = ["GaussianNB", "BernoulliNB", "MultinomialNB"]

classifiers = [GaussianNB(), BernoulliNB(), MultinomialNB()]

count = 0

for nome, clf in zip(names, classifiers): #zip itera sobre múltiplas listas simultaneamente
  model = clf
  #treinando o modelo
  model.fit(X_train, y_train)

  #predição
  y_pred = model.predict(X_test) #armazena as previsões do modelo
  proba = model.predict_proba(X_test) #armazena as probabilidades previstas para cada classe.

  #Resultados do classificador
  print(nome)
  # print(proba)
  print(classification_report(y_test, y_pred))
  show_decision_region(X_test, y_test, model)

GaussianNB
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        12
           1       0.93      0.87      0.90        15
           2       0.83      0.91      0.87        11

    accuracy                           0.92        38
   macro avg       0.92      0.93      0.92        38
weighted avg       0.92      0.92      0.92        38



BernoulliNB
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        12
           1       0.00      0.00      0.00        15
           2       0.29      1.00      0.45        11

    accuracy                           0.29        38
   macro avg       0.10      0.33      0.15        38
weighted avg       0.08      0.29      0.13        38



MultinomialNB
              precision    recall  f1-score   support

           0       1.00      0.83      0.91        12
           1       0.00      0.00      0.00        15
           2       0.39      1.00      0.56        11

    accuracy                           0.55        38
   macro avg       0.46      0.61      0.49        38
weighted avg       0.43      0.55      0.45        38

